# Active subspace of a CNN on BloodMNIST

<a href="https://colab.research.google.com/github/fatemehsalmani75/active-subspace-bloodmnist/blob/main/notebooks/bloodmnist_active_subspace.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook runs the whole pipeline:

1. train VGG-19 on BloodMNIST (8 classes of blood cells, 64x64),
2. compute the active subspace at a few layers and look at the eigenvalue spectrum,
3. build ASNet (active-subspace layer + polynomial chaos expansion) and compare size / accuracy,
4. build a universal perturbation from the active subspace and compare it with UAP.

Methods from Cui et al., *Active Subspace of Neural Networks: Structural Analysis and Universal Attacks*,
SIAM J. Math. Data Sci. 2020, and Moosavi-Dezfooli et al., *Universal Adversarial Perturbations*, CVPR 2017.
Data: MedMNIST v2 (Yang et al. 2023), CC BY 4.0.

Runtime: use a GPU (Runtime → Change runtime type → T4 GPU). With the default settings everything takes about 30–40 min.

In [ ]:
%cd /content
!test -d active-subspace-bloodmnist || git clone -q https://github.com/fatemehsalmani75/active-subspace-bloodmnist.git
%cd /content/active-subspace-bloodmnist
!git pull -q
!pip install -q medmnist adversarial-robustness-toolbox

In [ ]:
import copy
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from asnet.data import NUM_CLASSES, get_datasets, make_loader, subset
from asnet.models import build_model, count_params, split_model
from asnet.utils import set_seed, get_device, train_classifier, evaluate, stack_dataset
from asnet.active_subspace import compute_active_subspace, choose_rank
from asnet.asnet import build_asnet, finetune_asnet
from asnet.attack import as_universal_attack, fooling_rate, predict
from asnet.uap_baseline import uap_baseline

set_seed(0)
device = get_device()
print(device)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

## Settings

In [ ]:
SIZE = 64            # BloodMNIST image size
EPOCHS = 20          # baseline training
CUTS = [11, 16, 20]  # layers where the spectrum is computed (units of VGG-19)
CUT = 16             # layer used for ASNet (after the 4th pooling)
R, P = 50, 2         # active neurons, PCE degree
ASNET_EPOCHS = 10
AS_SAMPLES = 2000
FIT_SAMPLES = 4000
N_ATTACK = 200       # training images used to build the perturbations
DELTAS = [2, 4, 6, 8, 10]

## Data

In [ ]:
from medmnist import INFO
class_names = list(INFO["bloodmnist"]["label"].values())

train_aug, _, _ = get_datasets(SIZE, augment=True)
train, val, test = get_datasets(SIZE)
train_loader = make_loader(train_aug, 128, shuffle=True)
val_loader, test_loader = make_loader(val, 256), make_loader(test, 256)
print(len(train), len(val), len(test))

def show(img):
    return (img.permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5).clip(0, 1)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for c in range(NUM_CLASSES):
    idx = np.where(train.labels.reshape(-1) == c)[0][:2]
    for row in range(2):
        ax = axes[row, c]
        ax.imshow(show(train[idx[row]][0]))
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        if row == 0:
            ax.set_title(class_names[c][:18], fontsize=8)
plt.tight_layout(); plt.show()

## 1. Train VGG-19

In [ ]:
model = build_model("vgg19", NUM_CLASSES, SIZE)
print(f"{count_params(model):,} parameters")
model, val_acc = train_classifier(model, train_loader, val_loader, epochs=EPOCHS, device=device)
test_acc = evaluate(model, test_loader, device)
print(f"val {val_acc:.4f}  test {test_acc:.4f}")
torch.save(model.state_dict(), "vgg19_bloodmnist.pt")

## 2. Active subspace

For the cut after layer $l$ we look at the eigenvalues of $C = \mathbb{E}[\nabla_z \ell \, \nabla_z \ell^T]$.
If they decay fast, a few directions (active neurons) are enough to describe what the rest of the network does.

In [ ]:
as_loader = make_loader(subset(train, AS_SAMPLES, seed=0), 128)
x0 = next(iter(as_loader))[0].to(device)
model.eval()

spectra = {}
for cut in CUTS:
    pre, post, n = split_model(model, cut, x0)
    sub = compute_active_subspace(pre, post, as_loader, r=None, eps=0.05, sketch_size=200,
                                  device=device)
    # beyond the sketch size the eigenvalues are only rounding noise, so keep the first 150
    spectra[cut] = sub.eigvals.numpy()[:150]
    print(f"cut {cut:2d}: {n:6d} neurons, r(eps=0.05) = {sub.r}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
for cut, lam in spectra.items():
    a1.semilogy(np.arange(1, len(lam) + 1), lam / lam[0], label=f"layer {cut}")
    a2.plot(np.arange(1, len(lam) + 1), np.cumsum(lam) / lam.sum(), label=f"layer {cut}")
a1.set(xlabel="index i", ylabel=r"$\lambda_i / \lambda_1$", title="eigenvalue spectrum")
a2.axhline(0.95, ls="--", c="gray", lw=1)
a2.set(xlabel="number of directions r", ylabel="captured energy", title="cumulative energy", xlim=(0, 100))
a1.legend(); a2.legend()
plt.tight_layout(); plt.savefig("spectrum.png", bbox_inches="tight"); plt.show()

## 3. ASNet

In [ ]:
fit_loader = make_loader(subset(train, FIT_SAMPLES, seed=1), 256)
asnet, sub = build_asnet(model, CUT, as_loader, fit_loader, r=R, p=P, device=device)
init_acc = evaluate(asnet, test_loader, device)
print(f"after least-squares init: {init_acc:.4f}")

finetune_asnet(asnet, copy.deepcopy(model), train_loader, epochs=ASNET_EPOCHS, device=device)
asnet_acc = evaluate(asnet, test_loader, device)

summary = pd.DataFrame({
    "model": ["VGG-19", f"ASNet (cut {CUT}, r={R}, p={P})"],
    "test acc": [test_acc, asnet_acc],
    "parameters": [count_params(model), count_params(asnet)],
})
summary["compression"] = summary["parameters"][0] / summary["parameters"]
summary

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.5))
a1.bar(summary["model"], summary["parameters"] / 1e6, color=["#888", "#3b6ea5"])
a1.set(ylabel="parameters (M)", title="model size")
a2.bar(summary["model"], summary["test acc"] * 100, color=["#888", "#3b6ea5"])
a2.set(ylabel="test accuracy (%)", title="accuracy", ylim=(50, 100))
for a in (a1, a2):
    a.tick_params(axis="x", labelsize=8)
plt.tight_layout(); plt.savefig("asnet.png", bbox_inches="tight"); plt.show()

## 4. Universal attacks

One perturbation $v$ with $\|v\|_2 \le \delta$ is computed from 200 training images and then added to every test image.
Fooling rate = fraction of images whose predicted class changes.

In [ ]:
for p in model.parameters():
    p.requires_grad_(False)

Xtr, ytr = stack_dataset(subset(train, N_ATTACK, seed=2), device=device)
Xte, yte = stack_dataset(test, device=device)

rows, perts = [], {}
for d in DELTAS:
    res = as_universal_attack(model, Xtr, ytr, delta=d, log=lambda *a: None)
    v_uap = uap_baseline(model, Xtr, NUM_CLASSES, delta=d, max_iter=10, device=device)
    perts[d] = (res.v, v_uap)
    for name, v in [("active subspace", res.v), ("UAP", v_uap)]:
        rows.append({"delta": d, "method": name,
                     "train": fooling_rate(model, Xtr, v),
                     "test": fooling_rate(model, Xte, v)})
    print(f"delta={d}: AS {rows[-2]['test']:.3f}  UAP {rows[-1]['test']:.3f}")

attack_df = pd.DataFrame(rows)
attack_df.pivot(index="delta", columns="method", values=["train", "test"]).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, split in zip(axes, ["train", "test"]):
    for name, mk in [("active subspace", "o-"), ("UAP", "s--")]:
        d = attack_df[attack_df.method == name]
        ax.plot(d.delta, d[split] * 100, mk, label=name)
    ax.set(xlabel=r"$\|v\|_2$", title=f"fooling rate ({split})")
axes[0].set_ylabel("%")
axes[0].legend()
plt.tight_layout(); plt.savefig("attack.png", bbox_inches="tight"); plt.show()

### What the perturbations look like

In [ ]:
d = DELTAS[len(DELTAS) // 2]
v_as, v_uap = perts[d]

def vis(v):
    v = v[0].permute(1, 2, 0).cpu().numpy()
    return (v - v.min()) / (v.max() - v.min() + 1e-12)

clean_pred = predict(model, Xte[:6])
adv_pred = predict(model, (Xte[:6] + v_as).clamp(-1, 1))

fig, axes = plt.subplots(2, 7, figsize=(15, 4.6))
axes[0, 0].imshow(vis(v_as)); axes[0, 0].set_title("AS perturbation", fontsize=9)
axes[1, 0].imshow(vis(v_uap)); axes[1, 0].set_title("UAP perturbation", fontsize=9)
for i in range(6):
    axes[0, i + 1].imshow(show(Xte[i]))
    axes[0, i + 1].set_title(class_names[clean_pred[i]][:16], fontsize=8)
    axes[1, i + 1].imshow(show((Xte[i:i + 1] + v_as).clamp(-1, 1)[0]))
    axes[1, i + 1].set_title(class_names[adv_pred[i]][:16], fontsize=8,
                             color="crimson" if adv_pred[i] != clean_pred[i] else "black")
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
fig.suptitle(f"top: clean images, bottom: with the AS perturbation (delta = {d})")
plt.tight_layout(); plt.savefig("examples.png", bbox_inches="tight"); plt.show()

In [ ]:
# per-class fooling rate on the test set
v = perts[d][0]
adv = predict(model, (Xte + v).clamp(-1, 1))
clean = predict(model, Xte)
per_class = [(adv[yte == c] != clean[yte == c]).float().mean().item() for c in range(NUM_CLASSES)]

plt.figure(figsize=(9, 3.5))
plt.bar(range(NUM_CLASSES), np.array(per_class) * 100, color="#3b6ea5")
plt.xticks(range(NUM_CLASSES), [n[:14] for n in class_names], rotation=30, ha="right", fontsize=8)
plt.ylabel("fooling rate (%)"); plt.title(f"AS attack per class, delta = {d}")
plt.tight_layout(); plt.show()

In [ ]:
summary.to_csv("asnet_summary.csv", index=False)
attack_df.to_csv("attack_results.csv", index=False)